<a href="https://colab.research.google.com/github/Udaykiran606/Zepto-AI-ML-Capstone/blob/main/data_pipeline/data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""Data_Pipeline.py

Module 1 — Data Pipeline (books.toscrape.com)
Scrape -> Clean -> Convert -> Store -> Query -> Validate
"""

import sqlite3  # Used to create and manage the SQLite database
import time  # Used to pause between requests so we don't overload the server
from bs4 import BeautifulSoup  # Used to search and parse through HTML web pages
import numpy as np  # Used for handling missing values (NaN)
import pandas as pd  # Used to organize data into tables and save as CSV
import requests  # Used to download web pages from the internet

# The main website address we want to scrape
BASE_URL = "http://books.toscrape.com/"

# Project-defined fixed baseline conversion rate for GBP to INR
GBP_TO_INR_RATE = 105.50

# Database file name for the relational database step
DB_NAME = "catalog_pipeline.db"


def get_soup(url):
    """Downloads a webpage, forces UTF-8 encoding to prevent symbol errors (£),
    and turns it into a BeautifulSoup object.
    """
    response = requests.get(url)
    response.raise_for_status()  # Stops if the website returns an HTTP error
    response.encoding = "utf-8"  # Prevent special character (£) glitches
    return BeautifulSoup(response.text, "html.parser")


def scrape_category_books(category_url, category_name, all_books_data):
    """Navigates through a category page, extracts all books, and handles pagination."""
    current_url = category_url

    while current_url:
        print(f"-> Visiting page: {current_url}")
        soup = get_soup(current_url)
        books = soup.find_all("article", class_="product_pod")

        for book in books:
            # 1. Grab raw Title text
            title_elem = book.h3.a
            title = title_elem["title"] if title_elem else None

            # 2. Grab raw Price text
            price_elem = book.find("p", class_="price_color")
            price_raw = price_elem.text.strip() if price_elem else None

            # 3. Grab raw Star Rating text (e.g., 'Three')
            rating_p = book.find("p", class_="star-rating")
            star_rating_raw = rating_p["class"][1] if rating_p else None

            # 4. Grab raw Availability text
            avail_elem = book.find("p", class_="instock availability")
            availability_raw = avail_elem.text.strip() if avail_elem else None

            # Save raw extracted fields into our list
            all_books_data.append({
                "title": title,
                "price_raw": price_raw,
                "star_rating_raw": star_rating_raw,
                "availability_raw": availability_raw,
                "category": category_name,
            })

        # Check for pagination (Next page)
        next_button = soup.find("li", class_="next")
        if next_button and next_button.find("a"):
            next_page_relative = next_button.find("a")["href"]
            current_url = requests.compat.urljoin(current_url, next_page_relative)
        else:
            current_url = None

        time.sleep(0.3)


def clean_and_transform_data(df):
    """Cleans raw scraped data into precise types:
    - price_gbp (float): Strips currency symbols, handles parsing errors with median.
    - price_inr (float): Enriched column converted via fixed baseline rate (1 GBP = 105.50 INR).
    - rating (int): Maps word text ('One'–'Five') to integers (1–5), handles errors with median.
    - in_stock (bool): Parses availability text to True/False.
    - Drops rows where critical text fields (title) are missing.
    """
    print("\n--- Starting Data Cleaning & Transformation ---")

    # 1. Clean Title and Category (Drop rows if title is missing)
    df = df.dropna(subset=["title"])
    df["category"] = df["category"].str.strip().str.lower()

    # 2. Clean Price: Strip '£', 'Â£', or extra characters and convert to float
    def parse_price(val):
        try:
            cleaned = str(val).replace("£", "").replace("Â£", "").strip()
            return float(cleaned)
        except (ValueError, TypeError):
            return np.nan

    df["price_gbp"] = df["price_raw"].apply(parse_price)

    # Median-imputation for corrupted/missing prices
    price_median = df["price_gbp"].median()
    df["price_gbp"] = df["price_gbp"].fillna(price_median)
    print(f"-> Imputed missing/corrupt prices with median value: £{price_median:.2f}")

    # 3. Currency Enrichment: Convert price_gbp to price_inr using fixed baseline rate
    df["price_inr"] = round(df["price_gbp"] * GBP_TO_INR_RATE, 2)
    print(
        f"-> Enriched prices to INR using fixed baseline rate: 1 GBP = {GBP_TO_INR_RATE} INR"
    )

    # 4. Clean Star Rating: Map words to integers (1-5)
    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

    def parse_rating(val):
        return rating_map.get(val, np.nan)

    df["rating"] = df["star_rating_raw"].apply(parse_rating)

    # Median-imputation for corrupted/missing ratings
    rating_median = df["rating"].median()
    df["rating"] = df["rating"].fillna(rating_median).astype(int)
    print(f"-> Imputed missing/corrupt ratings with median value: {int(rating_median)}")

    # 5. Clean Availability: Parse text into boolean in_stock
    def parse_availability(val):
        if pd.isna(val):
            return False
        if "in stock" in str(val).lower():
            return True
        return False

    df["in_stock"] = df["availability_raw"].apply(parse_availability)

    # 6. Keep only pipeline-ready columns including price_inr
    cleaned_df = df[
        ["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]
    ].copy()

    return cleaned_df


def load_to_sqlite(df):
    """Creates a normalized SQLite database with 'categories' and 'books' tables
    sharing a Primary Key / Foreign Key relationship, then inserts the cleaned data.
    """
    print(f"\n--- Loading Data into Normalized SQLite Database ({DB_NAME}) ---")

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()

        # Drop existing tables to avoid locks and schema conflicts on clean re-runs
        cursor.execute("DROP TABLE IF EXISTS books")
        cursor.execute("DROP TABLE IF EXISTS categories")

        # 1. Create normalized tables schema
        cursor.execute("""
            CREATE TABLE categories (
                category_id INTEGER PRIMARY KEY AUTOINCREMENT,
                category_name TEXT UNIQUE NOT NULL
            )
        """)

        cursor.execute("""
            CREATE TABLE books (
                book_id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                price_gbp REAL NOT NULL,
                price_inr REAL NOT NULL,
                rating INTEGER NOT NULL,
                in_stock INTEGER NOT NULL,
                category_id INTEGER,
                FOREIGN KEY (category_id) REFERENCES categories(category_id)
            )
        """)

        # 2. Extract unique categories and insert them into the 'categories' table
        unique_categories = df["category"].unique()
        category_id_map = {}

        for cat_name in unique_categories:
            cursor.execute(
                "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
                (cat_name,),
            )
            cursor.execute(
                "SELECT category_id FROM categories WHERE category_name = ?",
                (cat_name,),
            )
            cat_id = cursor.fetchone()[0]
            category_id_map[cat_name] = cat_id

        # 3. Insert books into the 'books' table using mapped category_id foreign keys
        inserted_count = 0
        for _, row in df.iterrows():
            cat_id = category_id_map[row["category"]]
            in_stock_int = 1 if row["in_stock"] else 0

            cursor.execute(
                """
                INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    row["title"],
                    row["price_gbp"],
                    row["price_inr"],
                    row["rating"],
                    in_stock_int,
                    cat_id,
                ),
            )
            inserted_count += 1

        conn.commit()

    print(
        f"✅ Successfully loaded {inserted_count} books and "
        f"{len(unique_categories)} categories into SQLite!"
    )


def run_queries_and_validation():
    """Executes required SQL queries demonstrating SELECT/WHERE, ORDER BY, LIMIT,
    DISTINCT, BETWEEN/IN, and JOIN, reads them back into pandas, and validates
    join output equivalence with pandas pd.merge().
    """
    print("\n--- Executing Required SQL Queries & Validations ---")

    with sqlite3.connect(DB_NAME) as conn:
        # 1. SELECT / WHERE query
        query_where = (
            "SELECT title, price_gbp FROM books WHERE rating = 5 AND in_stock = 1"
        )
        df_where = pd.read_sql(query_where, conn)
        print(f"\n[Query 1: SELECT/WHERE] -> Found {len(df_where)} in-stock 5-star books.")
        print(df_where.head(3))

        # 2. ORDER BY & LIMIT query
        query_order_limit = "SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 5"
        df_order_limit = pd.read_sql(query_order_limit, conn)
        print("\n[Query 2: ORDER BY & LIMIT] -> Top 5 most expensive books in INR:")
        print(df_order_limit)

        # 3. DISTINCT query
        query_distinct = "SELECT DISTINCT rating FROM books ORDER BY rating DESC"
        df_distinct = pd.read_sql(query_distinct, conn)
        print("\n[Query 3: DISTINCT] -> Unique ratings present in catalog:")
        print(df_distinct)

        # 4. BETWEEN / IN query
        query_between = "SELECT title, price_gbp FROM books WHERE price_gbp BETWEEN 20.0 AND 40.0"
        df_between = pd.read_sql(query_between, conn)
        print(f"\n[Query 4: BETWEEN] -> Books priced between £20 and £40: {len(df_between)} found.")
        print(df_between.head(3))

        # 5. JOIN query (List books with their category names)
        query_join = """
            SELECT b.title, c.category_name, b.price_gbp, b.rating
            FROM books b
            JOIN categories c ON b.category_id = c.category_id
            ORDER BY b.rating DESC, b.price_gbp ASC
        """
        df_sql_join = pd.read_sql(query_join, conn)
        print(f"\n[Query 5: JOIN] -> Joined books with categories total rows: {len(df_sql_join)}")
        print(df_sql_join.head(3))

        # --- Validation Step: Reproduce JOIN via pandas pd.merge() ---
        print("\n--- Validating SQL JOIN with Pandas pd.merge() ---")
        df_books_raw = pd.read_sql("SELECT * FROM books", conn)
        df_cats_raw = pd.read_sql("SELECT * FROM categories", conn)

        # Perform equivalent merge in pandas
        df_pandas_merged = pd.merge(
            df_books_raw, df_cats_raw, on="category_id", how="inner"
        )
        df_pandas_merged = df_pandas_merged[
            ["title", "category_name", "price_gbp", "rating"]
        ]
        df_pandas_merged = df_pandas_merged.sort_values(
            by=["rating", "price_gbp"], ascending=[False, True]
        ).reset_index(drop=True)
        df_sql_join_reset = df_sql_join.reset_index(drop=True)

        # Check equivalence
        are_equivalent = df_sql_join_reset.equals(df_pandas_merged)
        print(f"✅ Pandas pd.merge() output matches SQL JOIN output? {are_equivalent}")


def main():
    print("--- Starting the Scraping Process ---")

    main_soup = get_soup(BASE_URL)
    sidebar = main_soup.find("div", class_="side_categories")
    category_links = sidebar.ul.ul.find_all("a")

    # Scrape first 8 categories to comfortably exceed the >= 60 books requirement
    target_categories = category_links[:8]
    all_books_data = []

    for cat in target_categories:
        cat_name = cat.text.strip()
        cat_relative_url = cat["href"]
        cat_full_url = requests.compat.urljoin(BASE_URL, cat_relative_url)

        print(f"\n[Category]: {cat_name}")
        scrape_category_books(cat_full_url, cat_name, all_books_data)

    # Convert raw data into initial Pandas DataFrame
    raw_df = pd.DataFrame(all_books_data)
    print(f"\n🎉 Successfully scraped raw data for {len(raw_df)} books.")

    # Acceptance Criteria Check
    if len(raw_df) < 60:
        print(f"⚠️ Warning: Scraped {len(raw_df)} books. Need at least 60!")
    else:
        print(f"✅ Criterion Met: Total books ({len(raw_df)}) >= 60.")

    # Run the cleaning and transformation pipeline
    final_df = clean_and_transform_data(raw_df)

    print(f"\n✨ Cleaned dataset ready! Total rows: {len(final_df)}")

    # Save finalized clean data to CSV as backup artifact
    final_df.to_csv("scraped_books_cleaned.csv", index=False)
    print("\n📁 Cleaned data saved successfully to 'scraped_books_cleaned.csv'.")

    # Load the cleaned dataset into the normalized SQLite database
    load_to_sqlite(final_df)

    # Run required SQL queries and cross-validate with pandas
    run_queries_and_validation()


if __name__ == "__main__":
    main()

--- Starting the Scraping Process ---

[Category]: Travel
-> Visiting page: http://books.toscrape.com/catalogue/category/books/travel_2/index.html

[Category]: Mystery
-> Visiting page: http://books.toscrape.com/catalogue/category/books/mystery_3/index.html
-> Visiting page: http://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html

[Category]: Historical Fiction
-> Visiting page: http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
-> Visiting page: http://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html

[Category]: Sequential Art
-> Visiting page: http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
-> Visiting page: http://books.toscrape.com/catalogue/category/books/sequential-art_5/page-2.html
-> Visiting page: http://books.toscrape.com/catalogue/category/books/sequential-art_5/page-3.html
-> Visiting page: http://books.toscrape.com/catalogue/category/books/sequential-art_5/page-4.ht